# CU7 Performance Assessment — LangganKu (starter)
Fill in your name and the last four digits of your IC, then work through the Tasks of the paper **in this notebook**.
First create your own data file in a terminal: `python make_my_data.py <L4>`  (the file `subscriber_churn.csv` appears in this folder).
Keep the notebook runnable from top to bottom.

In [54]:
import warnings; warnings.filterwarnings("ignore")
import mlflow, numpy as np, pandas as pd

NAME = "CHUAH SEONG JUN"      # e.g. "AISYAH BINTI AHMAD"
L4   = "0507"                # last four digits of your IC, keep the leading zero

d = [int(c) for c in L4]
# ---- your personal values (rules in the paper) ----
SEED      = int(4)
TEST_SIZE = 0.20 if d[3] % 2 == 0 else 0.25
DEPTH     = 3 + d[1] % 4
C1        = round(0.04 * (d[2] +1),2)
C2        = 10 * C1
RANK      = f"test_recall"
PORT      = 5100 + 7
EXP       = f"LangganKu_Churn_{L4}"
REG       = f"langganku_churn_{L4}"

print(f"SEED={SEED}  TEST_SIZE={TEST_SIZE}  DEPTH={DEPTH}  C1={C1} C2={C2} PORT={PORT} RANK={RANK} ")
print(f"EXP : {EXP}")
print(f"REG : {REG}")

SEED=4  TEST_SIZE=0.25  DEPTH=4  C1=0.04 C2=0.4 PORT=5107 RANK=test_recall 
EXP : LangganKu_Churn_0507
REG : langganku_churn_0507


## Task 1 — Set-up, split and pipeline

In [36]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

mlflow.set_tracking_uri(f"sqlite:///langganku_{L4}.db")
print(f"Tracking URL : sqlite:///langganku_{L4}.db")

df = pd.read_csv("subscriber_churn.csv")
print(f"Dataset      :{df.shape[0]} rows x {df.shape[1]} columns")
print(df.head(3))

X = df.drop(columns=["subscriber_id" , "churned"])
y= df["churned"]
print(f"\nClass balance — 0: {(y==0).sum()}  1: {(y==1).sum()}")

X_train , X_test , y_train , y_test = train_test_split(
    X , y ,
    test_size=TEST_SIZE,
    random_state=SEED
)
print(f"Train rows   : {len(X_train)}   Test rows : {len(X_test)}")

cat_cols = X.select_dtypes(exclude="number").columns.tolist()
print(f"Cat columns : {cat_cols}")

def make_pipeline(model):
    pre = ColumnTransformer(
        [("cat" , OneHotEncoder(handle_unknown="ignore") , cat_cols)],
        remainder="passthrough"
    )
    return Pipeline([("prep" , pre) , ("model", model)])

print("\n✓ Pipeline factory defined — Task 1 complete.")

Tracking URL : sqlite:///langganku_0507.db
Dataset      :1200 rows x 10 columns
  subscriber_id  monthly_fee_rm     plan  tenure_months contract  \
0         S0002            55.0  Premium             28   Yearly   
1         S0003            60.2  Premium             19  Monthly   
2         S0004            22.4    Basic              4  Monthly   

   support_calls payment_method  avg_hours_week main_device  churned  
0              3  Bank transfer            14.6       Phone        0  
1              2  Bank transfer            23.2      Laptop        0  
2              3  Bank transfer            11.1      Tablet        0  

Class balance — 0: 940  1: 260
Train rows   : 900   Test rows : 300
Cat columns : ['plan', 'contract', 'payment_method', 'main_device']

✓ Pipeline factory defined — Task 1 complete.


## Task 2 — Track the candidates in ONE experiment

In [40]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score , precision_score , recall_score , f1_score
from mlflow import MlflowClient

mlflow.set_experiment(EXP)
print(f"Experiment : {EXP}\n")

candidates = [
    (
        f"logreg-base-{L4}",
        LogisticRegression(max_iter=1000),
        {"max_iter": 1000},
        "linear"
    ),
    (
        f"logreg-c{C1:g}-{L4}",
        LogisticRegression(C=C1 , max_iter=1000),
        {"max_iter": 1000},
        "linear"
    ),
    (
        f"logreg-c{C2:g}-{L4}",
        LogisticRegression(C=C2 , max_iter=1000),
        {"max_iter": 1000},
        "linear"
    ),
    (
        f"rf-d{DEPTH}-{L4}",
        RandomForestClassifier(n_estimators=100 , max_depth=DEPTH, random_state=SEED),
        {"n_estimators": 100, "max_depth": DEPTH},
        "tree_ensemble"
    ),
]

for run_name , model , extra_params , model_type in candidates:
    with mlflow.start_run(run_name=run_name):

        mlflow.log_param("test_size", TEST_SIZE)
        mlflow.log_param("split_seed", SEED)
        mlflow.log_param("model",      type(model).__name__)
        for k, v in extra_params.items():
            mlflow.log_param(k, v)

        mlflow.set_tag("model_type", model_type)

        pipe = make_pipeline(model)
        pipe.fit(X_train , y_train)

        ytr_pred = pipe.predict(X_train)
        yte_pred = pipe.predict(X_test)

        metrics = {
            "train_accuracy":  float(accuracy_score(y_train,  ytr_pred)),
            "train_precision": float(precision_score(y_train, ytr_pred, zero_division=0)),
            "train_recall":    float(recall_score(y_train,    ytr_pred)),
            "train_f1":        float(f1_score(y_train,        ytr_pred)),
            "test_accuracy":   float(accuracy_score(y_test,   yte_pred)),
            "test_precision":  float(precision_score(y_test,  yte_pred, zero_division=0)),
            "test_recall":     float(recall_score(y_test,     yte_pred)),
            "test_f1":         float(f1_score(y_test,         yte_pred)),
        }
        mlflow.log_metrics(metrics)


        mlflow.sklearn.log_model(
            pipe, "model",
            skops_trusted_types=["sklearn.tree._tree.Tree"]
        )

        print(f"{run_name:44s}  "
              f"test_f1={metrics['test_f1']:.4f}  "
              f"test_acc={metrics['test_accuracy']:.4f}")

print("\n✓ Task 2 complete — 3 runs logged.")

Experiment : LangganKu_Churn_0507



2026/10/20 11:31:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


logreg-base-0507                              test_f1=0.5185  test_acc=0.8267


2026/10/20 11:32:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


logreg-c0.04-0507                             test_f1=0.4565  test_acc=0.8333


2026/10/20 11:33:14 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


logreg-c0.4-0507                              test_f1=0.5283  test_acc=0.8333


2026/10/20 11:33:48 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


rf-d4-0507                                    test_f1=0.2778  test_acc=0.8267

✓ Task 2 complete — 3 runs logged.


## Task 3 — Choose the best run from code and justify it

In [55]:
# your code
client   = MlflowClient()
exp_info = client.get_experiment_by_name(EXP)

all_runs = client.search_runs(
    experiment_ids=[exp_info.experiment_id],
    filter_string="status = 'FINISHED'",
    order_by=["metrics.test_f1 DESC"]
)

print("Rank  Run name                                       test_f1   test_acc")
print("-" * 74)
for i, r in enumerate(all_runs, 1):
    rname = r.data.tags.get("mlflow.runName", "?")
    f1    = r.data.metrics.get("test_f1",       0)
    acc   = r.data.metrics.get("test_accuracy", 0)
    print(f"  {i}   {rname:44s}  {f1:.4f}   {acc:.4f}")

best_run  = all_runs[0]
best_name = best_run.data.tags.get("mlflow.runName")
best_f1   = best_run.data.metrics["test_f1"]
best_acc  = best_run.data.metrics["test_accuracy"]

print(f"\n Champion  : {best_name}")
print(f"   test_f1   : {best_f1:.4f}")
print(f"   test_acc  : {best_acc:.4f}")
print(f"   run_id    : {best_run.info.run_id}")

Rank  Run name                                       test_f1   test_acc
--------------------------------------------------------------------------
  1   logreg-c0.4-0507                              0.5283   0.8333
  2   logreg-base-0507                              0.5185   0.8267
  3   logreg-c0.04-0507                             0.4565   0.8333
  4   rf-d4-0507                                    0.2778   0.8267

 Champion  : logreg-c0.4-0507
   test_f1   : 0.5283
   test_acc  : 0.8333
   run_id    : 0b82906f40744a0b8663ab9e7ddbef97


**Why I chose this run (2–3 sentences):**

## Task 4 — Register, alias and roll back

In [42]:
# your code

def find_run_by_name(name):
    for r in all_runs:
        if r.data.tags.get("mlflow.runName") == name:
            return r
    raise ValueError(f"Run '{name}' not found.")

baseline_name = f"logreg-base-{L4}"
baseline_run  = find_run_by_name(baseline_name)


v1 = mlflow.register_model(
    model_uri=f"runs:/{baseline_run.info.run_id}/model",
    name=REG
)
print(f"✓ Baseline registered  → version {v1.version}  (must be 1)")


v2 = mlflow.register_model(
    model_uri=f"runs:/{best_run.info.run_id}/model",
    name=REG
)
print(f"✓ Champion registered  → version {v2.version}")


client.set_registered_model_alias(REG, "champion", v2.version)
print(f"✓ Alias 'champion'     → version {v2.version}")


client.set_model_version_tag(REG, v2.version, "approved_by", NAME)
client.update_model_version(
    name=REG,
    version=v2.version,
    description=(
        f"Champion run: {best_name}. "
        f"Selected by highest test_f1 = {best_f1:.4f} (test_acc = {best_acc:.4f}). "
        f"Approved by: {NAME}."
    )
)
print(f"✓ Tag 'approved_by' and description set on version {v2.version}")


print("\n--- Roll-back demo ---")
client.set_registered_model_alias(REG, "champion", v1.version)
print(f"  [ROLLBACK] 'champion' → version {v1.version}  (baseline, safe fallback)")

client.set_registered_model_alias(REG, "champion", v2.version)
print(f"  [RESTORE]  'champion' → version {v2.version}  (champion restored)")


final = client.get_model_version_by_alias(REG, "champion")
print(f"\nFinal state: 'champion' alias = version {final.version}")
print("✓ Task 4 complete.")

Successfully registered model 'langganku_churn_0507'.
2026/10/20 11:34:20 WARNING mlflow.tracking._model_registry.fluent: Run with id 0fdc9df0abe8473985169091d7ed8818 has no artifacts at artifact path 'model', registering model based on models:/m-16f231f92de34a168a912ff67197fb01 instead
Created version '1' of model 'langganku_churn_0507'.
Registered model 'langganku_churn_0507' already exists. Creating a new version of this model...
2026/10/20 11:34:20 WARNING mlflow.tracking._model_registry.fluent: Run with id 0b82906f40744a0b8663ab9e7ddbef97 has no artifacts at artifact path 'model', registering model based on models:/m-3e1f5c2232d2440fbbe78f162d691947 instead


✓ Baseline registered  → version 1  (must be 1)


Created version '2' of model 'langganku_churn_0507'.


✓ Champion registered  → version 2
✓ Alias 'champion'     → version 2
✓ Tag 'approved_by' and description set on version 2

--- Roll-back demo ---
  [ROLLBACK] 'champion' → version 1  (baseline, safe fallback)
  [RESTORE]  'champion' → version 2  (champion restored)

Final state: 'champion' alias = version 2
✓ Task 4 complete.


## Task 5 — Serve the champion and call it

In [45]:
import subprocess, os, time

env = os.environ.copy()
env["MLFLOW_TRACKING_URI"] = f"sqlite:///LangganKu_{L4}.db"

serve_cmd = [
    "mlflow", "models", "serve",
    "-m", f"models:/{REG}@champion",
    "-p", str(PORT),
    "--no-conda",
]

print("Command:", " ".join(serve_cmd))
server_proc = subprocess.Popen(serve_cmd, env=env,
                               stdout=subprocess.PIPE, stderr=subprocess.PIPE)
print(f"\nServer PID {server_proc.pid} started on port {PORT}.")
print("Waiting 12 s for the server to become ready...")
time.sleep(12)
print("Ready — now run Cell 5-B to call it.")

Command: mlflow models serve -m models:/langganku_churn_0507@champion -p 5107 --no-conda

Server PID 16216 started on port 5107.
Waiting 12 s for the server to become ready...
Ready — now run Cell 5-B to call it.


In [47]:
import requests, json


my_row_idx = int(L4) % len(X)
my_row     = X.iloc[[my_row_idx]]

payload = {
    "dataframe_split": json.loads(
        my_row.to_json(orient="split", index=False)
    )
}

url = f"http://127.0.0.1:{PORT}/invocations"
print(f"POST → {url}")
print(f"Row index : {my_row_idx}")
print(my_row.to_string())

try:
    resp = requests.post(url, json=payload, timeout=15)
    print(f"\nHTTP status     : {resp.status_code}")
    pred = int(np.array(resp.json()["predictions"]).ravel()[0])
    label = "DEFAULT" if pred == 1 else "NOT default"
    print(f"Predicted class : {pred}  ({label})")
    print("\n✓ Task 5 complete — server answered correctly.")
except requests.exceptions.ConnectionError:
    print("\nx  Connection refused — server not ready yet.")
    print("   Wait a few seconds and re-run this cell only.")
except Exception as e:
    print(f"\nx  Error: {e}")

POST → http://127.0.0.1:5107/invocations
Row index : 507
     monthly_fee_rm   plan  tenure_months contract  support_calls payment_method  avg_hours_week main_device
507            23.1  Basic              7  Monthly              0   Cash voucher            16.9       Phone

HTTP status     : 200
Predicted class : 0  (NOT default)

✓ Task 5 complete — server answered correctly.
